# Week 4: Consumer Complaint Patterns and Response Trends

**AnalystLab Africa · Experience Lab Internship · Data Science Track** · **Author:** Molemi Molokoane

**Dataset:** CFPB consumer complaints about Bank of America, 1 May 2017 to 28 August 2023 (`Consumer_Complaints.xlsx`, 62,516 records, and `ConsumerComplaints_DataDictionary.csv`), supplied by AnalystLab Africa.

**Research question:** *How have complaint volumes, products, recorded company responses and CFPB forwarding intervals changed between 2017 and 2023, and which of these differences are statistically supported?*

> A complaint is an issue *reported by a consumer*. It does not establish misconduct, and a recorded company response is not proof that a complaint was resolved satisfactorily. All findings below describe recorded data only.

### Workflow
1. Business understanding and hypotheses
2. Data inspection
3. Data preparation and cleaning log
4. Exploratory data analysis
5. Hypothesis tests (H1 to H4)
6. Validation
7. Limitations and alternative explanations
8. Findings and recommendations

## 1. Business understanding

Bank of America's complaint and compliance teams need to know where consumers report problems, whether that is changing, and how recorded responses and timeliness look over time. This analysis supports the shared Week 4 platform: the cleaned dataset and KPI definitions produced here are the common inputs for the dashboard, forecasting and AI-assistant tracks.

### Hypotheses

| | Question | Null hypothesis (H₀) | Test |
|---|---|---|---|
| **H1** | Did monthly complaint volume change across years? | Monthly volumes have the same distribution in every year; there is no monotonic trend | Kruskal-Wallis on full-month counts; Mann-Kendall trend test |
| **H2** | Is the recorded company response associated with the product? | Response category is independent of product | Chi-square test of independence, Cramér's V |
| **H3** | Did the timely-response rate change over time? | The timely share is the same in every year | Chi-square test with standardised residuals |
| **H4** | Do CFPB forwarding intervals differ by product? | Forwarding intervals have the same distribution for every product | Kruskal-Wallis with ε²; chi-square on the same-day share |

With about 62,000 records, even trivial differences will be statistically significant, so every test is reported with an **effect size** and interpreted on that basis. A 5% significance level is used, with Holm correction wherever several comparisons are made.

### Data interpretation rules (from the brief)
- `Date submitted` = the date the CFPB **received** the complaint. `Date received` = the date the CFPB **sent** it to the company.
- Their difference is the **CFPB forwarding interval**, not the company's response or resolution time.
- A blank `Timely response?` is **not** a "No". In this data it only occurs on complaints still marked *In progress*.
- The timely-response rate uses only Yes/No records in the denominator; excluded records are reported separately.

### Success criteria
- Every brief data-preparation step is done and documented, and the raw file is provably unchanged.
- Each hypothesis is tested with assumptions checked, and interpreted with an effect size.
- Key figures reconcile with an independent recalculation.
- At least three findings are supported by evidence, with limitations stated.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path.cwd()
sys.path.insert(0, str(PROJECT))
import complaint_prep as cp

DATA = PROJECT / "Data"
RAW_FILE = DATA / "Consumer_Complaints.xlsx"
DICT_FILE = DATA / "ConsumerComplaints_DataDictionary.csv"
CLEAN_DIR = DATA / "Cleaned"
OUT = PROJECT / "Outputs"
VIS = PROJECT / "Visualizations"
for folder in (CLEAN_DIR, OUT, VIS):
    folder.mkdir(exist_ok=True)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

## 2. Data inspection

### 2.1 Load the raw file and record its fingerprint
The SHA-256 hash is recorded before any work and checked again at the end of the notebook, to show the original dataset is preserved.

In [2]:
RAW_HASH_BEFORE = cp.file_sha256(RAW_FILE)
print("SHA-256:", RAW_HASH_BEFORE)
print("Worksheets:", pd.ExcelFile(RAW_FILE).sheet_names)

raw = cp.load_raw(RAW_FILE)
print(f"Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

SHA-256: FCD270456EB9363482DA7136CF7915A4ED21FF705F31EFF995A0807B85B5E5F6


Worksheets: ['Data']


Shape: 62,516 rows x 12 columns


,Complaint ID,Submitted via,Date submitted,Date received,State,Product,Sub-product,Issue,Sub-issue,Company public response,Company response to consumer,Timely response?
0,4848023,Referral,2021-10-24,2021-10-27,NY,Mortgage,Conventional home mortgage,Applying for a mortgage or refinancing an existing mortgage,NaN,Company has responded to the consumer and the CFPB and chooses not to provid...,Closed with explanation,Yes
1,3621464,Web,2020-04-24,2020-04-24,FL,"Money transfer, virtual currency, or money service",Refund anticipation check,Lost or stolen check,NaN,Company has responded to the consumer and the CFPB and chooses not to provid...,Closed with monetary relief,Yes
2,5818349,Web,2022-07-27,2022-07-27,CA,"Credit reporting, credit repair services, or other personal consumer reports",Credit reporting,Incorrect information on your report,Account information incorrect,Company has responded to the consumer and the CFPB and chooses not to provid...,Closed with explanation,Yes
3,7233015,Referral,2023-07-10,2023-07-11,CA,Credit card or prepaid card,General-purpose prepaid card,Problem getting a card or closing an account,"Trouble getting, activating, or registering a card",NaN,In progress,NaN
4,5820224,Referral,2022-07-27,2022-07-28,VA,Credit card or prepaid card,General-purpose credit card or charge card,Closing your account,Company closed your account,Company has responded to the consumer and the CFPB and chooses not to provid...,Closed with explanation,Yes


### 2.2 Structure, column types and the data dictionary

In [3]:
dictionary = pd.read_csv(DICT_FILE)
structure = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "non_null": raw.notna().sum(),
    "missing": raw.isna().sum(),
    "missing_pct": (100 * raw.isna().mean()).round(2),
    "unique": raw.nunique(),
})
structure["in_dictionary"] = structure.index.isin(dictionary["Field"])
structure

,dtype,non_null,missing,missing_pct,unique,in_dictionary
Complaint ID,int64,62516,0,0.00,62516,True
Submitted via,str,62516,0,0.00,7,True
Date submitted,datetime64[us],62516,0,0.00,2308,True
Date received,datetime64[us],62516,0,0.00,2305,True
State,str,62516,0,0.00,51,True
Product,str,62516,0,0.00,9,True
Sub-product,str,62509,7,0.01,46,True
Issue,str,62516,0,0.00,76,True
Sub-issue,str,51658,10858,17.37,158,True
Company public response,str,60341,2175,3.48,6,True


All 12 columns match the brief and the data dictionary. Both date columns load as true dates and `Complaint ID` as an integer. The dictionary has a small typo (*"oossible"* in the Sub-issue description), which does not affect the analysis.

### 2.3 Uniqueness of Complaint ID and duplicate records

In [4]:
content_dup = cp.content_duplicate_mask(raw)
dup_groups = raw[content_dup].groupby([c for c in raw.columns if c != "Complaint ID"], dropna=False).size()
print("Complaint ID unique:           ", raw["Complaint ID"].is_unique)
print("Exact duplicate rows:          ", raw.duplicated().sum())
print(f"Rows identical except for ID:   {content_dup.sum():,} in {len(dup_groups):,} groups "
      f"(group size: median {dup_groups.median():.0f}, max {dup_groups.max()})")
raw.loc[content_dup, "Product"].value_counts().to_frame("rows in duplicate groups")

Complaint ID unique:            True


Exact duplicate rows:           0
Rows identical except for ID:   2,784 in 1,290 groups (group size: median 2, max 9)


,rows in duplicate groups
Product,
"Credit reporting, credit repair services, or other personal consumer reports",978
Checking or savings account,750
Credit card or prepaid card,567
Debt collection,215
Mortgage,193
"Money transfer, virtual currency, or money service",79
Vehicle loan or lease,2


**Decision: keep the 2,784 look-alike rows.** They have distinct complaint IDs, and the dataset has no consumer identifier or narrative, only a date, state and a few categories. Two different consumers in the same state reporting the same product and issue on the same day will produce identical rows, especially in high-volume categories such as credit reporting. Removing them would undercount complaints. They are flagged (`content_duplicate`) so their influence can be checked in the validation section.

### 2.4 Missing values and what they mean

In [5]:
in_progress = raw["Company response to consumer"].eq(cp.IN_PROGRESS)
missing = pd.DataFrame({
    "missing": raw.isna().sum(),
    "of_which_in_progress": raw[in_progress].isna().sum(),
}).query("missing > 0")
missing["of_which_closed"] = missing["missing"] - missing["of_which_in_progress"]
missing

,missing,of_which_in_progress,of_which_closed
Sub-product,7,0,7
Sub-issue,10858,160,10698
Company public response,2175,1494,681
Timely response?,1494,1494,0


In [6]:
print("Blank 'Timely response?' rows that are In progress:",
      raw["Timely response?"].isna().equals(in_progress))
print("In-progress complaints submitted between",
      raw.loc[in_progress, "Date submitted"].min().date(), "and",
      raw.loc[in_progress, "Date submitted"].max().date())
pd.crosstab(raw["Timely response?"].fillna("(blank)"), raw["Company response to consumer"], margins=True)

Blank 'Timely response?' rows that are In progress: True
In-progress complaints submitted between 2023-05-12 and 2023-08-28


Company response to consumer,Closed,Closed with explanation,Closed with monetary relief,Closed with non-monetary relief,In progress,All
Timely response?,,,,,,
(blank),0,0,0,0,1494,1494
No,8,1527,690,178,0,2403
Yes,0,39517,14007,5095,0,58619
All,8,41044,14697,5273,1494,62516


- **Timely response?** All 1,494 blanks are *In progress* complaints, all submitted after 12 May 2023. They are labelled *Pending* and kept out of the timely-rate denominator.
- **Company public response:** 1,494 blanks are In progress (no response yet). The other 681 are closed complaints with no public response recorded. The public response is optional, so these are labelled *Not recorded*, not treated as errors.
- **Sub-issue** (10,858) and **Sub-product** (7): the dictionary says not every issue has a sub-issue and not every product a sub-product. Blanks are labelled, not imputed.

In [7]:
raw.loc[raw["Sub-product"].isna(), ["Product", "Issue"]]

,Product,Issue
5623,Checking or savings account,Incorrect information on your report
14554,Checking or savings account,Problem with a credit reporting company's investigation into an existing pro...
39878,Checking or savings account,Incorrect information on your report
40582,Checking or savings account,Problem with a credit reporting company's investigation into an existing pro...
42790,Checking or savings account,Incorrect information on your report
55010,Checking or savings account,Incorrect information on your report
62288,Checking or savings account,Incorrect information on your report


The 7 rows without a sub-product are all checking or savings accounts reporting *credit report* issues, which look like miscategorised complaints. They are too few to matter and are kept as filed.

### 2.5 Date validation

In [8]:
fwd = (raw["Date received"] - raw["Date submitted"]).dt.days
date_checks = pd.Series({
    "earliest Date submitted": raw["Date submitted"].min().date(),
    "latest Date submitted": raw["Date submitted"].max().date(),
    "earliest Date received": raw["Date received"].min().date(),
    "latest Date received": raw["Date received"].max().date(),
    "rows with a time component": int(((raw["Date submitted"] != raw["Date submitted"].dt.normalize()) |
                                       (raw["Date received"] != raw["Date received"].dt.normalize())).sum()),
    "forwarded before submitted (negative interval)": int((fwd < 0).sum()),
    "forwarded the same day": int((fwd == 0).sum()),
    f"interval > {cp.LONG_FORWARDING_DAYS} days": int((fwd > cp.LONG_FORWARDING_DAYS).sum()),
    "longest interval (days)": int(fwd.max()),
}, name="value").to_frame()
date_checks

,value
earliest Date submitted,2017-05-01
latest Date submitted,2023-08-28
earliest Date received,2017-05-01
latest Date received,2023-08-28
rows with a time component,0
forwarded before submitted (negative interval),0
forwarded the same day,49634
interval > 30 days,435
longest interval (days),275


Dates fall exactly in the stated range, and no complaint was forwarded before it was received. Intervals above 30 days (435 complaints) are plausible operational delays, so they are flagged for analysis, not removed.

### 2.6 Reporting months: gaps and partial periods

In [9]:
month = raw["Date submitted"].dt.to_period("M")
per_month = raw.groupby(month)["Date submitted"].agg(
    complaints="size", first_day="min", last_day="max",
    days_with_complaints=lambda s: s.dt.day.nunique())
all_months = pd.period_range(per_month.index.min(), per_month.index.max(), freq="M")
print(f"{len(per_month)} months with data out of {len(all_months)} in range; "
      f"missing months: {sorted(set(all_months) - set(per_month.index)) or 'none'}")
print(f"Monthly volume: median {per_month['complaints'].median():.0f}, "
      f"min {per_month['complaints'].min()}, max {per_month['complaints'].max()}")
pd.concat([per_month.head(2), per_month.tail(3)])

76 months with data out of 76 in range; missing months: none
Monthly volume: median 749, min 356, max 1749


,complaints,first_day,last_day,days_with_complaints
Date submitted,,,,
2017-05,744,2017-05-01,2017-05-31,31
2017-06,701,2017-06-01,2017-06-30,30
2023-06,1018,2023-06-01,2023-06-30,30
2023-07,1749,2023-07-01,2023-07-31,31
2023-08,717,2023-08-01,2023-08-28,26


- **August 2023** ends on the 28th, so it is a partial month.
- **May 2017** has complaints on all 31 days and a typical volume (744), so the data suggests it is complete. The brief treats the first and last months as partial, so **both are excluded from month-level tests** (H1). That costs only one possibly complete month.
- **July 2023** is the busiest month (1,749, against a median of 749). The daily counts below show whether this comes from one day or is spread across the month.

In [10]:
july = raw.loc[month == "2023-07", "Date submitted"].dt.date.value_counts().sort_index()
print(f"July 2023 daily complaints: median {july.median():.0f}, max {july.max()} on {july.idxmax()}")
july.sort_values(ascending=False).head(5).to_frame("complaints")

July 2023 daily complaints: median 53, max 183 on 2023-07-12


,complaints
Date submitted,
2023-07-12,183
2023-07-13,148
2023-07-11,108
2023-07-14,98
2023-07-15,81


The July 2023 peak is **not** spread across the month: 11 to 14 July alone account for 537 complaints, against a typical 53 a day. A short burst like this points to a one-off trigger (for example news coverage or a regulatory announcement concerning the bank) rather than a change in underlying service. The dataset cannot show the cause, so the month is **kept** as genuine recorded volume and treated as an outlier in interpretation, not removed.

### 2.7 Categorical consistency

In [11]:
category_checks = []
for col in cp.TEXT_COLUMNS:
    values = raw[col].dropna().astype(str)
    category_checks.append({
        "column": col,
        "distinct_values": values.nunique(),
        "with_extra_spaces": int((values != values.str.strip()).sum()),
        "case_or_space_variants": values.nunique() - values.str.strip().str.lower().nunique(),
    })
pd.DataFrame(category_checks).set_index("column")

,distinct_values,with_extra_spaces,case_or_space_variants
column,,,
Submitted via,7,0,0
State,51,0,0
Product,9,0,0
Sub-product,46,0,0
Issue,76,0,0
Sub-issue,158,0,0
Company public response,6,0,0
Company response to consumer,5,0,0
Timely response?,2,0,0


In [12]:
for col in ["Submitted via", "Product", "Company response to consumer", "Company public response", "Timely response?"]:
    display(raw[col].value_counts(dropna=False).to_frame(col))

,Submitted via
Submitted via,
Web,45423
Referral,10766
Phone,4684
Postal mail,1318
Fax,233
Web Referral,90
Email,2


,Product
Product,
Checking or savings account,24814
Credit card or prepaid card,16197
"Credit reporting, credit repair services, or other personal consumer reports",7710
Mortgage,6601
"Money transfer, virtual currency, or money service",3453
Debt collection,2736
Vehicle loan or lease,633
"Payday loan, title loan, or personal loan",333
Student loan,39


,Company response to consumer
Company response to consumer,
Closed with explanation,41044
Closed with monetary relief,14697
Closed with non-monetary relief,5273
In progress,1494
Closed,8


,Company public response
Company public response,
Company has responded to the consumer and the CFPB and chooses not to provide a public response,60311
NaN,2175
Company believes the complaint provided an opportunity to answer consumer's questions,12
Company believes the complaint is the result of a misunderstanding,7
Company believes it acted appropriately as authorized by contract or law,6
Company believes complaint caused principally by actions of third party outside the control or direction of the company,3
Company disputes the facts presented in the complaint,2


,Timely response?
Timely response?,
Yes,58619
No,2403
NaN,1494


No category has spelling, spacing or case variants, so no values need merging. Points worth noting:
- **Submitted via:** *Web Referral* (90) and *Email* (2) are small but legitimate CFPB channel values; they are kept as recorded.
- **Company response to consumer:** *Closed* without further detail has only 8 records, all marked untimely. It is kept as its own category.
- **Company public response:** 96.5% of records are *"chooses not to provide a public response"*. Only 30 records carry a substantive public statement, too few for statistical analysis.
- **State:** 51 values (50 states plus DC); no territories or invalid codes.

## 3. Data preparation

All cleaning is done by `complaint_prep.clean()`, which is covered by unit tests (`tests/test_complaint_prep.py`). It **keeps every row and every raw column unchanged** and adds cleaned or derived values as new snake_case columns, so each value can be traced back to the original.

In [13]:
df, cleaning_log = cp.clean(raw)
cleaning_log

,step,column,rule,rows_affected,rationale
0,1,all text columns,Trim leading and trailing spaces,0,Prevents the same category from being counted twice; no case or spacing vari...
1,2,forwarding_days,"Date received minus Date submitted, in whole days",62516,CFPB forwarding interval; not the company's response time. No negative inter...
2,3,long_forwarding,Flag intervals above 30 days,435,"Kept, not removed: long intervals are plausible and are analysed as a group"
3,4,"year, quarter, month, year_month, year_quarter",Derived from Date submitted,62516,Volume is dated by when the CFPB received the complaint
4,5,boundary_month,Flag the first (2017-05) and last (2023-08) months,1461,The brief treats them as partial; excluded from month-level tests only
5,6,product_short,Short chart label for each product,62516,Full CFPB product names are kept in Product
6,7,sub_product,Blank -> 'Not specified',7,The dictionary says not all products have sub-products
7,8,sub_issue,Blank -> 'No sub-issue recorded',10858,The dictionary says not all issues have sub-issues; a blank is not missing data
8,9,public_response,Blank and In progress -> 'Pending (complaint in progress)',1494,"The company has not finished responding, so no public response is expected yet"
9,10,public_response,Blank and closed -> 'Not recorded',681,A public response is optional; a blank is not a refusal or a dispute


In [14]:
derived = [c for c in df.columns if c not in cp.RAW_COLUMNS]
print(f"{len(df):,} rows, {len(cp.RAW_COLUMNS)} raw columns + {len(derived)} derived columns:")
print(", ".join(derived))
df[derived].head()

62,516 rows, 12 raw columns + 19 derived columns:
date_submitted, date_received, forwarding_days, same_day_forwarding, long_forwarding, year, quarter, month, year_month, year_quarter, boundary_month, product_short, sub_product, sub_issue, in_progress, public_response, timely_status, timely, content_duplicate


,date_submitted,date_received,forwarding_days,same_day_forwarding,long_forwarding,year,quarter,month,year_month,year_quarter,boundary_month,product_short,sub_product,sub_issue,in_progress,public_response,timely_status,timely,content_duplicate
0,2021-10-24,2021-10-27,3,False,False,2021,4,10,2021-10,2021Q4,False,Mortgage,Conventional home mortgage,No sub-issue recorded,False,Company has responded to the consumer and the CFPB and chooses not to provid...,Yes,True,False
1,2020-04-24,2020-04-24,0,True,False,2020,2,4,2020-04,2020Q2,False,Money transfer,Refund anticipation check,No sub-issue recorded,False,Company has responded to the consumer and the CFPB and chooses not to provid...,Yes,True,False
2,2022-07-27,2022-07-27,0,True,False,2022,3,7,2022-07,2022Q3,False,Credit reporting,Credit reporting,Account information incorrect,False,Company has responded to the consumer and the CFPB and chooses not to provid...,Yes,True,False
3,2023-07-10,2023-07-11,1,False,False,2023,3,7,2023-07,2023Q3,False,Credit/prepaid card,General-purpose prepaid card,"Trouble getting, activating, or registering a card",True,Pending (complaint in progress),Pending (in progress),<NA>,False
4,2022-07-27,2022-07-28,1,False,False,2022,3,7,2022-07,2022Q3,False,Credit/prepaid card,General-purpose credit card or charge card,Company closed your account,False,Company has responded to the consumer and the CFPB and chooses not to provid...,Yes,True,False


### 3.1 Validation after cleaning
These checks stop the notebook if any cleaning step changed a count it should not have.

In [15]:
checks = {
    "row count unchanged": len(df) == len(raw) == 62_516,
    "Complaint ID still unique": df["Complaint ID"].is_unique,
    "raw columns unchanged": all(df[c].equals(raw[c]) for c in cp.RAW_COLUMNS),
    "no blanks in derived columns (except timely)": df[[c for c in derived if c != "timely"]].notna().all().all(),
    "timely blanks == In progress": df["timely"].isna().equals(df["in_progress"]),
    "Yes + No + Pending == total": sum(cp.timely_rate(df)[k] for k in ("yes", "no", "excluded_pending")) == len(df),
    "product shares sum to 100%": abs(cp.share_table(df, "Product")["pct"].sum() - 100) < 0.05,
    "monthly counts sum to total": cp.monthly_counts(df).sum() == len(df),
    "annual counts sum to total": cp.yoy_change(df)["complaints"].sum() == len(df),
    "forwarding interval never negative": (df["forwarding_days"] >= 0).all(),
    "76 months, none missing": len(cp.monthly_counts(df)) == 76 and (cp.monthly_counts(df) > 0).all(),
}
validation = pd.Series(checks, name="passed").to_frame()
assert validation["passed"].all(), validation[~validation["passed"]]
validation

,passed
row count unchanged,True
Complaint ID still unique,True
raw columns unchanged,True
no blanks in derived columns (except timely),True
timely blanks == In progress,True
Yes + No + Pending == total,True
product shares sum to 100%,True
monthly counts sum to total,True
annual counts sum to total,True
forwarding interval never negative,True


### 3.2 Headline KPIs on the cleaned data
Shared definitions for all tracks (also exported to `Outputs/kpi_definitions.csv`).

In [16]:
timely = cp.timely_rate(df)
forwarding = cp.forwarding_summary(df)
kpi_definitions = pd.DataFrame([
    ("Total complaints", "Count of unique Complaint IDs", f"{len(df):,}"),
    ("Complaint volume period", "Dated by Date submitted (when the CFPB received the complaint)",
     f"{df['date_submitted'].min().date()} to {df['date_submitted'].max().date()}"),
    ("Timely-response rate", "Yes / (Yes + No); In-progress (blank) records excluded and reported separately",
     f"{timely['timely_pct']:.2f}% ({timely['yes']:,} of {timely['denominator']:,}; "
     f"{timely['excluded_pending']:,} pending excluded)"),
    ("In-progress complaints", "Company response to consumer = 'In progress'", f"{int(df['in_progress'].sum()):,}"),
    ("Forwarding interval", "Date received minus Date submitted, in days (CFPB to company; not company response time)",
     f"mean {forwarding['mean_days']:.2f} days, median {forwarding['median_days']:.0f}"),
    ("Same-day forwarding", "Share of complaints with a forwarding interval of 0 days", f"{forwarding['same_day_pct']:.1f}%"),
    ("Unusually long forwarding", f"Forwarding interval > {cp.LONG_FORWARDING_DAYS} days",
     f"{forwarding['long_count']:,} complaints"),
    ("Product share", "Complaints for a product / total complaints", "see Outputs/product_share.csv"),
    ("Year-over-year change", "Change in annual complaints; 2017 and 2023 are partial years, so per-month averages are also compared",
     "see Outputs/annual_volume.csv"),
], columns=["kpi", "definition", "value"]).set_index("kpi")
kpi_definitions

,definition,value
kpi,,
Total complaints,Count of unique Complaint IDs,"62,516"
Complaint volume period,Dated by Date submitted (when the CFPB received the complaint),2017-05-01 to 2023-08-28
Timely-response rate,Yes / (Yes + No); In-progress (blank) records excluded and reported separately,"96.06% (58,619 of 61,022; 1,494 pending excluded)"
In-progress complaints,Company response to consumer = 'In progress',"1,494"
Forwarding interval,"Date received minus Date submitted, in days (CFPB to company; not company re...","mean 1.22 days, median 0"
Same-day forwarding,Share of complaints with a forwarding interval of 0 days,79.4%
Unusually long forwarding,Forwarding interval > 30 days,435 complaints
Product share,Complaints for a product / total complaints,see Outputs/product_share.csv
Year-over-year change,"Change in annual complaints; 2017 and 2023 are partial years, so per-month a...",see Outputs/annual_volume.csv


In [17]:
cp.yoy_change(df)

,complaints,months,complaints_per_month,yoy_change_pct,yoy_change_per_month_pct,partial_year
year,,,,,,
2017,5394,8,674.2,NaN,NaN,True
2018,7872,12,656.0,45.9,-2.7,False
2019,7075,12,589.6,-10.1,-10.1,False
2020,8942,12,745.2,26.4,26.4,False
2021,11149,12,929.1,24.7,24.7,False
2022,12953,12,1079.4,16.2,16.2,False
2023,9131,8,1141.4,-29.5,5.7,True


### 3.3 Exports
- `Data/Cleaned/complaints_clean.csv`: the cleaned dataset (raw columns plus derived columns), for the dashboard, forecasting and AI-assistant tracks.
- `Outputs/cleaning_log.csv`, `Outputs/data_quality_summary.csv`, `Outputs/kpi_definitions.csv`, `Outputs/product_share.csv`, `Outputs/annual_volume.csv`.

In [18]:
df.to_csv(CLEAN_DIR / "complaints_clean.csv", index=False)
cleaning_log.to_csv(OUT / "cleaning_log.csv", index=False)
kpi_definitions.to_csv(OUT / "kpi_definitions.csv")
cp.share_table(df, "Product").to_csv(OUT / "product_share.csv")
cp.yoy_change(df).to_csv(OUT / "annual_volume.csv")

data_quality = pd.DataFrame([
    ("Records", len(raw), "Matches the brief (62,516)"),
    ("Columns", raw.shape[1], "Matches the brief and data dictionary"),
    ("Duplicate Complaint IDs", int(raw["Complaint ID"].duplicated().sum()), "Complaint ID is a valid primary key"),
    ("Exact duplicate rows", int(raw.duplicated().sum()), ""),
    ("Rows identical except Complaint ID", int(content_dup.sum()), "Kept and flagged; no consumer identifier exists"),
    ("Blank Sub-product", int(raw["Sub-product"].isna().sum()), "Labelled 'Not specified'"),
    ("Blank Sub-issue", int(raw["Sub-issue"].isna().sum()), "Labelled 'No sub-issue recorded'"),
    ("Blank public response (in progress)", int((raw["Company public response"].isna() & in_progress).sum()), "Labelled 'Pending'"),
    ("Blank public response (closed)", int((raw["Company public response"].isna() & ~in_progress).sum()), "Labelled 'Not recorded'"),
    ("Blank Timely response?", int(raw["Timely response?"].isna().sum()), "All In progress; excluded from timely rate"),
    ("Negative forwarding intervals", int((fwd < 0).sum()), ""),
    (f"Forwarding intervals > {cp.LONG_FORWARDING_DAYS} days", int((fwd > cp.LONG_FORWARDING_DAYS).sum()), "Kept and flagged"),
    ("Missing months in range", len(set(all_months) - set(per_month.index)), "76 of 76 months present"),
    ("Category spelling/case variants", int(sum(c["case_or_space_variants"] for c in category_checks)), "No standardisation needed"),
], columns=["check", "result", "action"])
data_quality.to_csv(OUT / "data_quality_summary.csv", index=False)
data_quality

,check,result,action
0,Records,62516,"Matches the brief (62,516)"
1,Columns,12,Matches the brief and data dictionary
2,Duplicate Complaint IDs,0,Complaint ID is a valid primary key
3,Exact duplicate rows,0,
4,Rows identical except Complaint ID,2784,Kept and flagged; no consumer identifier exists
5,Blank Sub-product,7,Labelled 'Not specified'
6,Blank Sub-issue,10858,Labelled 'No sub-issue recorded'
7,Blank public response (in progress),1494,Labelled 'Pending'
8,Blank public response (closed),681,Labelled 'Not recorded'
9,Blank Timely response?,1494,All In progress; excluded from timely rate
